In [1]:
# Experiment 4: Tokenizer lab - BPE vs WordPiece vs SentencePiece on English, Telugu and Hindi
from transformers import AutoTokenizer
from google.colab import userdata
from google import genai
import pandas as pd

texts = {
    "English": "Generative AI models predict the next token in a sentence.",
    "Telugu":  "జనరేటివ్ ఏఐ నమూనాలు వాక్యంలో తదుపరి టోకెన్‌ను అంచనా వేస్తాయి.",
    "Hindi":   "जनरेटिव एआई मॉडल वाक्य में अगले टोकन का अनुमान लगाते हैं।",
}
tokenizers = {
    "BPE (gpt2)": "gpt2",
    "WordPiece (bert-base-multilingual-cased)": "bert-base-multilingual-cased",
    "SentencePiece (xlm-roberta-base)": "xlm-roberta-base",
}

rows = []
for name, model_id in tokenizers.items():
    tok = AutoTokenizer.from_pretrained(model_id)
    for lang, s in texts.items():
        ids = tok(s, add_special_tokens=False).input_ids
        rows.append({"tokenizer": name, "language": lang, "words": len(s.split()),
                     "tokens": len(ids), "tokens_per_word": round(len(ids) / len(s.split()), 2)})
    print(name, "| vocab size:", tok.vocab_size)
    print("   English pieces:", tok.tokenize(texts["English"])[:10])
    print("   Telugu pieces :", tok.tokenize(texts["Telugu"])[:10])
    print("   Token IDs (Hindi):", tok(texts["Hindi"], add_special_tokens=False).input_ids[:10])

# Hosted model's own tokenizer (Gemini) for comparison
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
for lang, s in texts.items():
    n = client.models.count_tokens(model="gemini-3.1-flash-lite", contents=s).total_tokens
    rows.append({"tokenizer": "Gemini (gemini-3.1-flash-lite)", "language": lang, "words": len(s.split()),
                 "tokens": n, "tokens_per_word": round(n / len(s.split()), 2)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Token-ratio table (documented result): how many times more tokens than English
ratio = df.pivot(index="tokenizer", columns="language", values="tokens")
ratio["Telugu / English"] = (ratio["Telugu"] / ratio["English"]).round(2)
ratio["Hindi / English"] = (ratio["Hindi"] / ratio["English"]).round(2)
print("\nToken-ratio table:")
print(ratio.to_string())

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

BPE (gpt2) | vocab size: 50257
   English pieces: ['Gener', 'ative', 'ĠAI', 'Ġmodels', 'Ġpredict', 'Ġthe', 'Ġnext', 'Ġtoken', 'Ġin', 'Ġa']
   Telugu pieces : ['à', '°', 'ľ', 'à', '°', '¨', 'à', '°', '°', 'à']
   Token IDs (Hindi): [11976, 250, 11976, 101, 11976, 108, 24231, 229, 11976, 253]


... (6 outputs left out)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

SentencePiece (xlm-roberta-base) | vocab size: 250002
   English pieces: ['▁Genera', 'tive', '▁AI', '▁models', '▁predict', '▁the', '▁next', '▁to', 'ken', '▁in']
   Telugu pieces : ['▁జన', 'రే', 'టివ్', '▁ఏ', 'ఐ', '▁న', 'మూ', 'నాలు', '▁వా', 'క్']
   Token IDs (Hindi): [15103, 8503, 207628, 9489, 31046, 187543, 147131, 421, 127831, 197670]
                               tokenizer language  words  tokens  tokens_per_word
                              BPE (gpt2)  English     10      12             1.20
                              BPE (gpt2)   Telugu      8     166            20.75
                              BPE (gpt2)    Hindi     11      90             8.18
WordPiece (bert-base-multilingual-cased)  English     10      14             1.40
WordPiece (bert-base-multilingual-cased)   Telugu      8      33             4.12
WordPiece (bert-base-multilingual-cased)    Hindi     11      31             2.82
        SentencePiece (xlm-roberta-base)  English     10      13             1.30
    